In [ ]:
# If needed:
# !pip install xgboost -q

import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

from xgboost import XGBClassifier

BASE_DIR = "/content/drive/MyDrive/AML_Project_v2"

# Inputs
AE_FEATURES_PKL   = os.path.join(BASE_DIR, "ae_v2_features_0p2.pkl")
MV_EMB_MC_NPY     = os.path.join(BASE_DIR, "node_embeddings_mv_sage_mc_0p2.npy")
ACC_MAP_PKL       = os.path.join(BASE_DIR, "account_index_map_0p2.pkl")
ACC_UNCERT_NPY    = os.path.join(BASE_DIR, "account_uncertainty_mc_0p2.npy")
ACC_UNSTABLE_NPY  = os.path.join(BASE_DIR, "account_unstable_mask_mc_0p2.npy")

# Outputs (new model for MC-dropout version)
XGB_MODEL_PKL      = os.path.join(BASE_DIR, "xgboost_v5_ae_mvgnn_mc_0p2.pkl")
FUSION_SCHEMA_JSON = os.path.join(BASE_DIR, "xgb_fusion_schema_mc_0p2.json")

SEED = 42
np.random.seed(SEED)

print("BASE_DIR:", BASE_DIR)
print("AE_FEATURES_PKL  :", AE_FEATURES_PKL)
print("MV_EMB_MC_NPY    :", MV_EMB_MC_NPY)
print("ACC_MAP_PKL      :", ACC_MAP_PKL)
print("ACC_UNCERT_NPY   :", ACC_UNCERT_NPY)
print("ACC_UNSTABLE_NPY :", ACC_UNSTABLE_NPY)


BASE_DIR: /content/drive/MyDrive/AML_Project_v2
AE_FEATURES_PKL  : /content/drive/MyDrive/AML_Project_v2/ae_v2_features_0p2.pkl
MV_EMB_MC_NPY    : /content/drive/MyDrive/AML_Project_v2/node_embeddings_mv_sage_mc_0p2.npy
ACC_MAP_PKL      : /content/drive/MyDrive/AML_Project_v2/account_index_map_0p2.pkl
ACC_UNCERT_NPY   : /content/drive/MyDrive/AML_Project_v2/account_uncertainty_mc_0p2.npy
ACC_UNSTABLE_NPY : /content/drive/MyDrive/AML_Project_v2/account_unstable_mask_mc_0p2.npy


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df_ae = pd.read_pickle(AE_FEATURES_PKL)
print("AE features shape:", df_ae.shape)

print("\nColumns:")
print(df_ae.columns.tolist())

LABEL_COL   = "Is Laundering"
ACCOUNT_COL = "Account"

assert LABEL_COL in df_ae.columns
assert ACCOUNT_COL in df_ae.columns

print("\nLabel distribution:")
print(df_ae[LABEL_COL].value_counts())
display(df_ae.head())


AE features shape: (1015669, 68)

Columns:
['AE_z_0', 'AE_z_1', 'AE_z_2', 'AE_z_3', 'AE_z_4', 'AE_z_5', 'AE_z_6', 'AE_z_7', 'AE_z_8', 'AE_z_9', 'AE_z_10', 'AE_z_11', 'AE_z_12', 'AE_z_13', 'AE_z_14', 'AE_z_15', 'AE_z_16', 'AE_z_17', 'AE_z_18', 'AE_z_19', 'AE_z_20', 'AE_z_21', 'AE_z_22', 'AE_z_23', 'AE_z_24', 'AE_z_25', 'AE_z_26', 'AE_z_27', 'AE_z_28', 'AE_z_29', 'AE_z_30', 'AE_z_31', 'AE_z_32', 'AE_z_33', 'AE_z_34', 'AE_z_35', 'AE_z_36', 'AE_z_37', 'AE_z_38', 'AE_z_39', 'AE_z_40', 'AE_z_41', 'AE_z_42', 'AE_z_43', 'AE_z_44', 'AE_z_45', 'AE_z_46', 'AE_z_47', 'AE_z_48', 'AE_z_49', 'AE_z_50', 'AE_z_51', 'AE_z_52', 'AE_z_53', 'AE_z_54', 'AE_z_55', 'AE_z_56', 'AE_z_57', 'AE_z_58', 'AE_z_59', 'AE_z_60', 'AE_z_61', 'AE_z_62', 'AE_z_63', 'AE_recon_error', 'Is Laundering', 'Account', 'Timestamp']

Label distribution:
Is Laundering
0    1014582
1       1087
Name: count, dtype: int64


,AE_z_0,AE_z_1,AE_z_2,AE_z_3,AE_z_4,AE_z_5,AE_z_6,AE_z_7,AE_z_8,AE_z_9,...,AE_z_58,AE_z_59,AE_z_60,AE_z_61,AE_z_62,AE_z_63,AE_recon_error,Is Laundering,Account,Timestamp
0,-0.171639,-4.232361,0.104417,-3.298181,4.128160,1.148903,-4.008280,0.183923,4.746537,-1.055284,...,2.734672,-0.448340,-0.755428,-3.457057,-0.710143,1.532658,0.000016,0,80E50C3C0,2022/09/01 00:29
1,-0.552513,1.176918,0.085590,1.292784,2.337363,3.379844,-0.583980,-1.723102,0.499747,2.023540,...,1.297836,-0.107377,1.208097,-3.487659,-2.267819,0.238896,0.000018,0,8001C6CC0,2022/09/01 13:28
2,0.693883,2.977150,0.714605,1.302704,5.107666,1.099602,-2.010736,0.382961,6.933664,1.577050,...,2.689955,-2.235557,-1.524162,-6.733779,-1.529114,-3.113523,0.000078,0,80CAF3CE0,2022/09/01 02:46
3,-2.277668,1.706605,1.744195,-0.888375,2.416312,-2.294415,-2.257886,0.131887,5.121434,1.695682,...,-0.526143,1.533264,-0.255519,-5.008838,-2.726768,1.109005,0.000049,0,804DC2C20,2022/09/02 08:02
4,-0.396464,-0.740129,-2.468286,0.665525,1.697380,3.444777,-0.233828,-2.433388,0.482001,0.393666,...,4.090092,0.048419,-0.156657,-2.817705,-2.929527,0.502151,0.000062,0,80A5EC8A0,2022/09/09 18:01


In [ ]:
emb_mv_mc = np.load(MV_EMB_MC_NPY)      # (num_accounts, 128)
account_to_idx = joblib.load(ACC_MAP_PKL)

uncert_acc   = np.load(ACC_UNCERT_NPY)   # (num_accounts,)
unstable_mask = np.load(ACC_UNSTABLE_NPY)  # (num_accounts,)

print("MV MC emb shape       :", emb_mv_mc.shape)
print("Account uncert shape  :", uncert_acc.shape)
print("Unstable mask shape   :", unstable_mask.shape)

num_accounts, mv_dim = emb_mv_mc.shape


MV MC emb shape       : (331899, 128)
Account uncert shape  : (331899,)
Unstable mask shape   : (331899,)


In [ ]:
accounts = df_ae[ACCOUNT_COL].astype(str).values
n = len(df_ae)

# Map MV embedding per transaction
mv_features = np.zeros((n, mv_dim), dtype=np.float32)
uncert_feat = np.zeros((n, 1), dtype=np.float32)
unstable_feat = np.zeros((n, 1), dtype=np.float32)

missing = 0
for i, acc in enumerate(accounts):
    idx = account_to_idx.get(acc, None)
    if idx is not None:
        mv_features[i] = emb_mv_mc[idx]
        uncert_feat[i, 0] = uncert_acc[idx]
        unstable_feat[i, 0] = unstable_mask[idx]
    else:
        # unseen accounts → MV emb & uncertainty stay 0
        missing += 1

print("Transactions with missing account embedding:", missing)

# AE latent + recon error
ae_z_cols = [c for c in df_ae.columns if c.startswith("AE_z_")]
assert len(ae_z_cols) > 0

X_ae_latent = df_ae[ae_z_cols].values
X_ae_recon  = df_ae[["AE_recon_error"]].values

print("AE latent shape :", X_ae_latent.shape)
print("AE recon shape  :", X_ae_recon.shape)
print("MV features     :", mv_features.shape)
print("Uncert feature  :", uncert_feat.shape)
print("Unstable feature:", unstable_feat.shape)

# Final fusion matrix:
# [AE_z, AE_recon_error, MV_emb_MC, uncert_acc, unstable_mask]
X = np.hstack([X_ae_latent, X_ae_recon, mv_features, uncert_feat, unstable_feat])
y = df_ae[LABEL_COL].values.astype(int)

print("Final fusion X shape:", X.shape)
print("Labels shape       :", y.shape)
print("Fraud count        :", (y == 1).sum())


Transactions with missing account embedding: 0
AE latent shape : (1015669, 64)
AE recon shape  : (1015669, 1)
MV features     : (1015669, 128)
Uncert feature  : (1015669, 1)
Unstable feature: (1015669, 1)
Final fusion X shape: (1015669, 195)
Labels shape       : (1015669,)
Fraud count        : 1087


In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print("Train size:", X_train.shape[0])
print("Val size  :", X_val.shape[0])
print("Test size :", X_test.shape[0])

print("\nLabel dist (train):")
print(pd.Series(y_train).value_counts())
print("\nLabel dist (val):")
print(pd.Series(y_val).value_counts())
print("\nLabel dist (test):")
print(pd.Series(y_test).value_counts())


Train size: 710968
Val size  : 152350
Test size : 152351

Label dist (train):
0    710207
1       761
Name: count, dtype: int64

Label dist (val):
0    152187
1       163
Name: count, dtype: int64

Label dist (test):
0    152188
1       163
Name: count, dtype: int64


In [ ]:
# scale_pos_weight to handle imbalance
pos_count = (y_train == 1).sum()
neg_count = (y_train == 0).sum()
scale_pos_weight = neg_count / max(pos_count, 1)

print("Train positives:", pos_count)
print("Train negatives:", neg_count)
print("scale_pos_weight:", scale_pos_weight)

xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    scale_pos_weight=scale_pos_weight,
    random_state=SEED,
    n_jobs=-1,
)

eval_set = [(X_train, y_train), (X_val, y_val)]

xgb_model.fit(
    X_train, y_train,
    eval_set=eval_set,
    verbose=True,
)


Train positives: 761
Train negatives: 710207
scale_pos_weight: 933.2549277266754
[0]	validation_0-logloss:0.65699	validation_1-logloss:0.65700
[1]	validation_0-logloss:0.62430	validation_1-logloss:0.62434
[2]	validation_0-logloss:0.59446	validation_1-logloss:0.59454
[3]	validation_0-logloss:0.56719	validation_1-logloss:0.56729
[4]	validation_0-logloss:0.54285	validation_1-logloss:0.54303
[5]	validation_0-logloss:0.52019	validation_1-logloss:0.52030
[6]	validation_0-logloss:0.49921	validation_1-logloss:0.49935
[7]	validation_0-logloss:0.48053	validation_1-logloss:0.48064
[8]	validation_0-logloss:0.46251	validation_1-logloss:0.46264
[9]	validation_0-logloss:0.44614	validation_1-logloss:0.44629
[10]	validation_0-logloss:0.43110	validation_1-logloss:0.43125
[11]	validation_0-logloss:0.41731	validation_1-logloss:0.41750
[12]	validation_0-logloss:0.40393	validation_1-logloss:0.40410
[13]	validation_0-logloss:0.39018	validation_1-logloss:0.39037
[14]	validation_0-logloss:0.37898	validation_1-

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=400, n_jobs=-1,
              num_parallel_tree=None, ...)

In [ ]:
def evaluate_split(name, X_split, y_split, model):
    y_proba = model.predict_proba(X_split)[:, 1]
    y_pred  = (y_proba >= 0.5).astype(int)

    acc  = accuracy_score(y_split, y_pred)
    prec = precision_score(y_split, y_pred, zero_division=0)
    rec  = recall_score(y_split, y_pred, zero_division=0)
    f1   = f1_score(y_split, y_pred, zero_division=0)
    auc  = roc_auc_score(y_split, y_proba)

    print(f"\n===== {name} =====")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1       : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}")

    print("\nConfusion matrix:")
    print(confusion_matrix(y_split, y_pred))

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "roc_auc": auc,
    }

metrics_train = evaluate_split("TRAIN", X_train, y_train, xgb_model)
metrics_val   = evaluate_split("VAL",   X_val,   y_val,   xgb_model)
metrics_test  = evaluate_split("TEST",  X_test,  y_test,  xgb_model)

print("\n===== Classification Report (TEST) =====")
y_test_proba = xgb_model.predict_proba(X_test)[:, 1]
y_test_pred  = (y_test_proba >= 0.5).astype(int)
print(classification_report(y_test, y_test_pred, digits=4))



===== TRAIN =====
Accuracy : 0.9686
Precision: 0.0330
Recall   : 1.0000
F1       : 0.0639
ROC-AUC  : 0.9987

Confusion matrix:
[[687910  22297]
 [     0    761]]

===== VAL =====
Accuracy : 0.9669
Precision: 0.0195
Recall   : 0.6074
F1       : 0.0378
ROC-AUC  : 0.9600

Confusion matrix:
[[147204   4983]
 [    64     99]]

===== TEST =====
Accuracy : 0.9681
Precision: 0.0204
Recall   : 0.6135
F1       : 0.0395
ROC-AUC  : 0.9599

Confusion matrix:
[[147386   4802]
 [    63    100]]

===== Classification Report (TEST) =====
              precision    recall  f1-score   support

           0     0.9996    0.9684    0.9838    152188
           1     0.0204    0.6135    0.0395       163

    accuracy                         0.9681    152351
   macro avg     0.5100    0.7910    0.5116    152351
weighted avg     0.9985    0.9681    0.9828    152351



In [ ]:
# feature names in order:
mv_feature_names = [f"MV_emb_mc_{i}" for i in range(mv_dim)]

fusion_feature_names = (
    ae_z_cols
    + ["AE_recon_error"]
    + mv_feature_names
    + ["GNN_uncertainty", "GNN_unstable_flag"]
)

schema_mc = {
    "label_col": LABEL_COL,
    "account_col": ACCOUNT_COL,
    "ae_z_cols": ae_z_cols,
    "ae_recon_col": "AE_recon_error",
    "mv_dim": mv_dim,
    "mv_feature_names": mv_feature_names,
    "extra_features": ["GNN_uncertainty", "GNN_unstable_flag"],
    "fusion_feature_names": fusion_feature_names,
    "train_metrics": metrics_train,
    "val_metrics": metrics_val,
    "test_metrics": metrics_test,
    "mv_embedding_file": os.path.basename(MV_EMB_MC_NPY),
    "uncertainty_file": os.path.basename(ACC_UNCERT_NPY),
    "unstable_mask_file": os.path.basename(ACC_UNSTABLE_NPY),
    "model_name": "xgboost_v5_ae_mvgnn_mc_0p2",
}

joblib.dump(xgb_model, XGB_MODEL_PKL)
with open(FUSION_SCHEMA_JSON, "w") as f:
    json.dump(schema_mc, f, indent=2)

print("Saved MC XGBoost model →", XGB_MODEL_PKL)
print("Saved MC fusion schema →", FUSION_SCHEMA_JSON)


Saved MC XGBoost model → /content/drive/MyDrive/AML_Project_v2/xgboost_v5_ae_mvgnn_mc_0p2.pkl
Saved MC fusion schema → /content/drive/MyDrive/AML_Project_v2/xgb_fusion_schema_mc_0p2.json


In [ ]:
print("===== STAGE C2 (MC) SUMMARY =====")
print("Model path :", XGB_MODEL_PKL)
print("Schema path:", FUSION_SCHEMA_JSON)

print("\nTrain metrics:", metrics_train)
print("\nVal metrics:",   metrics_val)
print("\nTest metrics:",  metrics_test)


===== STAGE C2 (MC) SUMMARY =====
Model path : /content/drive/MyDrive/AML_Project_v2/xgboost_v5_ae_mvgnn_mc_0p2.pkl
Schema path: /content/drive/MyDrive/AML_Project_v2/xgb_fusion_schema_mc_0p2.json

Train metrics: {'accuracy': 0.9686385322546163, 'precision': 0.0330037297250412, 'recall': 1.0, 'f1': 0.06389856836978883, 'roc_auc': np.float64(0.9987422250809899)}

Val metrics: {'accuracy': 0.9668723334427306, 'precision': 0.01948051948051948, 'recall': 0.6073619631901841, 'f1': 0.03775023832221163, 'roc_auc': np.float64(0.9599559284527298)}

Test metrics: {'accuracy': 0.9680671607012753, 'precision': 0.02039983680130559, 'recall': 0.6134969325153374, 'f1': 0.039486673247778874, 'roc_auc': np.float64(0.9598874599885417)}


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/AML_Project_v2"

files_required = {
    "AE model (optional)": "autoencoder_model.pth",
    "AE features (0.2 dataset)": "ae_v2_features_0p2.pkl",
    "Graph - Account": "graph_accounts_0p2.pt",
    "Graph - Bank": "graph_banks_0p2.pt",
    "Account index map": "account_index_map_0p2.pkl",
    "Bank index map": "bank_index_map_0p2.pkl",
    "0.2 sampled dataset": "HI_Trans_0p2.csv",

    # Deterministic version (Stage C)
    "MV-GNN deterministic embeddings": "node_embeddings_mv_sage_0p2.npy",
    "XGB v4 model": "xgboost_v4_ae_mvgnn_0p2.pkl",
    "Fusion schema v4": "xgb_fusion_schema_0p2.json",

    # MC-Dropout version (Stage C2)
    "MV-GNN MC embeddings": "node_embeddings_mv_sage_mc_0p2.npy",
    "Account uncertainties MC": "account_uncertainty_mc_0p2.npy",
    "Account unstable mask MC": "account_unstable_mask_mc_0p2.npy",
    "XGB v5 model (MC)": "xgboost_v5_ae_mvgnn_mc_0p2.pkl",
    "Fusion schema v5": "xgb_fusion_schema_mc_0p2.json",
}

print("===== INTERFACE FILE CHECKER =====\n")

all_ok = True
for desc, fname in files_required.items():
    path = os.path.join(BASE_DIR, fname)
    exists = os.path.exists(path)

    print(f"{desc:35s}: {'OK ✓' if exists else 'MISSING ✗'}   --> {fname}")

    if not exists:
        all_ok = False

print("\n==================================")
print("ALL FILES FOUND!" if all_ok else "Some files are missing. Fix before interface.")


===== INTERFACE FILE CHECKER =====

AE model (optional)                : MISSING ✗   --> autoencoder_model.pth
AE features (0.2 dataset)          : OK ✓   --> ae_v2_features_0p2.pkl
Graph - Account                    : OK ✓   --> graph_accounts_0p2.pt
Graph - Bank                       : OK ✓   --> graph_banks_0p2.pt
Account index map                  : OK ✓   --> account_index_map_0p2.pkl
Bank index map                     : OK ✓   --> bank_index_map_0p2.pkl
0.2 sampled dataset                : OK ✓   --> HI_Trans_0p2.csv
MV-GNN deterministic embeddings    : OK ✓   --> node_embeddings_mv_sage_0p2.npy
XGB v4 model                       : OK ✓   --> xgboost_v4_ae_mvgnn_0p2.pkl
Fusion schema v4                   : OK ✓   --> xgb_fusion_schema_0p2.json
MV-GNN MC embeddings               : OK ✓   --> node_embeddings_mv_sage_mc_0p2.npy
Account uncertainties MC           : OK ✓   --> account_uncertainty_mc_0p2.npy
Account unstable mask MC           : OK ✓   --> account_unstable_mask_mc_0p2.n